# CAN/ISOBUS Fine-Tuning (Colab, free T4 GPU)

Fine-tunes a 7-8B open model on your `finetune_data.jsonl` QA pairs (generated locally by `dataset_prep.py`) using Unsloth + QLoRA.

**Before running:** in the menu above, go to `Runtime -> Change runtime type -> T4 GPU`, then `Runtime -> Run all`.

**What you need on hand:** the `finetune_data.jsonl` file produced by `dataset_prep.py` on your laptop -- you'll upload it in Cell 3.

In [ ]:
# Cell 1 -- confirm you actually got a GPU runtime
import torch
assert torch.cuda.is_available(), "No GPU detected -- go to Runtime > Change runtime type > T4 GPU, then re-run."
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Cell 2 -- install Unsloth (takes ~1-2 min)
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

In [ ]:
# Cell 3 -- upload finetune_data.jsonl from your laptop
from google.colab import files
uploaded = files.upload()  # pick finetune_data.jsonl in the dialog
DATA_FILE = list(uploaded.keys())[0]
print(f"Uploaded: {DATA_FILE}")

In [ ]:
# Cell 4 -- load and format the QA pairs as chat-style training text
from datasets import load_dataset

raw_dataset = load_dataset("json", data_files=DATA_FILE, split="train")
print(f"{len(raw_dataset)} QA pairs loaded")
print(raw_dataset[0])

In [ ]:
# Cell 5 -- load the base model in 4-bit
# Swap MODEL_NAME for any tag from https://huggingface.co/unsloth if you
# want a different size -- e.g. "unsloth/Llama-3.2-3B-Instruct" for a
# faster/smaller run, or a Qwen2.5-7B-Instruct variant for a bigger one.
from unsloth import FastLanguageModel
import torch

MODEL_NAME = "unsloth/Meta-Llama-3.1-8B-Instruct"
MAX_SEQ_LEN = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = MAX_SEQ_LEN,
    dtype = None,           # auto-detect best dtype for the T4
    load_in_4bit = True,    # ~70% VRAM reduction, this is the "QLoRA" part
)

In [ ]:
# Cell 6 -- format the dataset using this model's chat template, now that
# the tokenizer (with its template) is loaded
def format_example(example):
    messages = [
        {"role": "user", "content": example["question"]},
        {"role": "assistant", "content": example["answer"]},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}

dataset = raw_dataset.map(format_example)
print(dataset[0]["text"])

In [ ]:
# Cell 7 -- attach a LoRA adapter (only ~1% of parameters get trained,
# the base model's own weights stay frozen)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                       "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",  # Unsloth's memory-optimized version
    random_state = 3407,
)

In [ ]:
# Cell 8 -- train (roughly 15-40 min on a free T4 for ~500-800 examples,
# 3 epochs -- exact time depends on how busy Colab's shared GPUs are)
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = MAX_SEQ_LEN,
    args = SFTConfig(
        output_dir = "outputs",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        logging_steps = 10,
        save_strategy = "epoch",
        report_to = "none",
    ),
)

trainer.train()

In [ ]:
# Cell 9 -- save the LoRA adapter and download it as a zip
model.save_pretrained("can_isobus_lora")
tokenizer.save_pretrained("can_isobus_lora")

!zip -r can_isobus_lora.zip can_isobus_lora

from google.colab import files
files.download("can_isobus_lora.zip")

## Optional: export to GGUF for use back in LM Studio

Since your existing project already runs on LM Studio, exporting to GGUF
lets you load the fine-tuned model exactly like you load `gemma4:e4b` now --
no separate serving setup needed. This step takes a few extra minutes.

In [ ]:
# Cell 10 -- export a quantized GGUF file, downloadable straight into LM Studio
model.save_pretrained_gguf("can_isobus_gguf", tokenizer, quantization_method="q4_k_m")

import glob
gguf_path = glob.glob("can_isobus_gguf/*.gguf")[0]
print(f"GGUF file: {gguf_path}")

from google.colab import files
files.download(gguf_path)

## Next steps

1. In LM Studio: `My Models -> Import Model File`, point it at the downloaded `.gguf`.
2. Load it in the Local Server tab, same as `gemma4:e4b`.
3. Ask it the same questions you tested against your RAG pipeline, **without**
   giving it any retrieved context -- see how much it actually learned versus
   what RAG was supplying at question time. That comparison is the real payoff
   of doing both.